In [ ]:
from omni.vectorstore_queries import search_tweets

from omni.db import get_tweet_store

In [ ]:
store = get_tweet_store()

In [ ]:
author_screen_names = ["karpathy", "iamtrask", "miles_brundage"]

In [ ]:
query = store.search_documents()
query = query.where({"author.screen_name__in": author_screen_names})
documents = query.get()

In [ ]:
query_text = "RAG"
cosine_threshold = 0.0


In [ ]:
cosine_docs = store.search_chunks() \
    .where({"id__in": [doc.id for doc in documents]}) \
    .semantic(query_text, score_threshold=0.0) \
    .chunk_limit(4000) \
    .get_documents()

In [ ]:
cosine_scores = {}
for doc in cosine_docs:
    max_doc_score = max(chunk.score for chunk in doc.chunks)
    cosine_scores[doc.id] = max_doc_score

In [ ]:
len(documents)

In [ ]:
reranking_docs = (
    store.search_chunks()
    .where({"id__in": [doc.id for doc in documents]})
    .semantic(query_text, score_threshold=0.0) \
    .chunk_limit(1000)
    .rerank(rerank_query=query_text, score_threshold=0.0)
    .get_documents()
)

In [ ]:
reranking_scores = {}
for doc in reranking_docs:
    max_doc_score = max(chunk.score for chunk in doc.chunks)
    reranking_scores[doc.id] = max_doc_score

In [ ]:
scaler = sum(xi/yi for xi, yi in zip(cosine_scores.values(), reranking_scores.values())) / len(cosine_scores)

In [ ]:
scaler

In [ ]:
import math

mean_x = sum(x) / len(x)
std_x = math.sqrt(sum((xi - mean_x) ** 2 for xi in x) / len(x))
std_x


In [ ]:
import matplotlib.pyplot as plt

# Find common document ids between cosine_scores and reranking_scores
common_ids = set(cosine_scores.keys()) & set(reranking_scores.keys())

x = [cosine_scores[doc_id] for doc_id in common_ids]
y = [reranking_scores[doc_id] for doc_id in common_ids]

x1 = list(i/10 for i in range(0, 10))
y1 = [xi/0.42 for xi in x1]

# plt.legend()

# plt.plot(x1,y1)


plt.figure(figsize=(6, 6))
# plt.scatter(x, y, c='blue', label='Docs', alpha=0.7)
# plt.scatter(x, [0]*len(x), c='red', label='Cosine Score', alpha=0.5)
# # Plot each (x, y) pair as a single dot
# plt.scatter(x, y, c='purple', label='Cosine vs Reranking', alpha=0.8)
plt.plot(x1, y1, c='orange', label='y=0.42x')
plt.scatter(x, y, c='purple', label='Cosine vs Reranking', alpha=0.8)

plt.xlim(min(x)-0.05, max(x)+0.05)
plt.ylim(min(y)-0.05, max(y)+0.05)

plt.xlabel("Cosine Score")
plt.ylabel("Reranking Score")
plt.title("Cosine vs Reranking Scores")
plt.grid(True)
plt.show()


In [ ]:
res = search_tweets(query_text="Hallucinate",
    author_screen_names= ["karpathy", "iamtrask", "miles_brundage"],
    cosine_threshold= 0.37,
    reranking_threshold= 0.81,
    limit= 50,
) 


In [ ]:
len(res)

In [ ]:
for x in res[:5]:
    print(x.author["name"], x.similarity_score)
    print(x.content)
    print()